# Correr simulaciones con estimador `argmin_pyomo`

Este notebook corre simulaciones Monte Carlo usando el estimador del centro seleccionado con Pyomo. Guarda las salidas en `results/data/` sin sobrescribir los CSV principales.

Archivos de salida por defecto:

- `tn_pyomo_simulation_raw_quick.csv`
- `tn_pyomo_simulation_summary_quick.csv`
- `sn_pyomo_simulation_raw_quick.csv`
- `sn_pyomo_simulation_summary_quick.csv`

Requisitos: instalar `pyomo` y un solver compatible. La implementacion prueba `appsi_highs`, `highs`, `glpk` y `cbc`; para HiGHS normalmente basta instalar `highspy`.

In [1]:
# --- Setup: imports y path al repo ---
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.distributions import default_h0_specs, default_ha_specs
from src.pyomo_argmin import DEFAULT_PYOMO_SOLVERS
from src.simulation import SimConfig, summarize
from src.simulation_sn import SimConfigSn, summarize_sn

print(f'Working dir: {Path.cwd()}')
print(f'CPU cores disponibles: {os.cpu_count()}')
print(f'Solvers Pyomo que se probaran: {DEFAULT_PYOMO_SOLVERS}')

Working dir: c:\Users\juanc\github\Proyecto2EstadisticaNP
CPU cores disponibles: 12
Solvers Pyomo que se probaran: ('appsi_highs', 'highs', 'glpk', 'cbc')


## Validar Pyomo y solver

Ejecuta esta celda antes de lanzar simulaciones largas. Si falla, instala `pyomo` y un solver.

In [2]:
def check_pyomo_ready():
    try:
        import pyomo.environ as pyo
    except ImportError as exc:
        raise RuntimeError(
            'Falta Pyomo. Instala pyomo y un solver, por ejemplo: pip install pyomo highspy'
        ) from exc

    available = []
    for solver in DEFAULT_PYOMO_SOLVERS:
        opt = pyo.SolverFactory(solver)
        if opt.available(exception_flag=False):
            available.append(solver)

    if not available:
        raise RuntimeError(
            'Pyomo esta instalado, pero no hay solver disponible. '
            'Instala highspy, GLPK o CBC.'
        )

    print(f'Pyomo listo. Solvers disponibles: {available}')
    return available

available_solvers = check_pyomo_ready()

Pyomo listo. Solvers disponibles: ['appsi_highs', 'highs']


## Parametros

Por defecto se usa una corrida pequena. Pyomo se invoca dentro de cada replica bootstrap, asi que subir `B`, `R`, pesos o tamanos puede aumentar mucho el tiempo.

In [3]:
# --- Que correr ---
run_tn = True
run_sn = True
use_parallel = True
quick = True

# --- Distribuciones ---
specs = default_h0_specs() + default_ha_specs()

if quick:
    # B=39 da alpha*(B+1)=2 para alpha=0.05.
    tn_config = SimConfig(
        sample_sizes=(20, 40),
        estimators=('argmin_pyomo',),
        B=39,
        R=20,
        alpha=0.05,
        seed=2026,
    )
    sn_config = SimConfigSn(
        sample_sizes=(20, 40),
        estimators=('argmin_pyomo',),
        qs=(2,),
        weight_names=('gauss_1.0',),
        B=39,
        R=10,
        alpha=0.05,
        seed=2026,
        n_t_grid=101,
    )
    suffix = '_quick'
else:
    tn_config = SimConfig(
        sample_sizes=(20, 40, 80),
        estimators=('argmin_pyomo',),
        B=199,
        R=100,
        alpha=0.05,
        seed=2026,
    )
    sn_config = SimConfigSn(
        sample_sizes=(20, 40, 80),
        estimators=('argmin_pyomo',),
        qs=(2,),
        weight_names=('gauss_1.0', 'laplace_1.0'),
        B=199,
        R=60,
        alpha=0.05,
        seed=2026,
        n_t_grid=201,
    )
    suffix = ''

print(f'Distribuciones: {[s.name for s in specs]}')
print(f'T_n config: n={tn_config.sample_sizes}, estimators={tn_config.estimators}, B={tn_config.B}, R={tn_config.R}')
print(f'S_n config: n={sn_config.sample_sizes}, estimators={sn_config.estimators}, q={sn_config.qs}, pesos={sn_config.weight_names}, B={sn_config.B}, R={sn_config.R}, K={sn_config.n_t_grid}')

Distribuciones: ['Uniforme(1.0,3.0)', 'Cauchy(loc=2.0,scale=1.0)', 'Gamma(k=2.0,s=1.0)', 'Weibull(k=1.5,s=1.0)', 'Pareto(a=3.0,s=1.0)']
T_n config: n=(20, 40), estimators=('argmin_pyomo',), B=39, R=20
S_n config: n=(20, 40), estimators=('argmin_pyomo',), q=(2,), pesos=('gauss_1.0',), B=39, R=10, K=101


## Ejecutar `T_n` con `argmin_pyomo`

In [4]:
tn_df = None
tn_summary = None

if run_tn:
    if use_parallel:
        from src.simulation_parallel import run_simulation_parallel
        n_workers = max(1, (os.cpu_count() or 2) - 1)
        tn_df = run_simulation_parallel(specs, tn_config, n_workers=n_workers)
    else:
        from src.simulation import run_simulation
        tn_df = run_simulation(specs, tn_config)

    tn_summary = summarize(tn_df, alpha=tn_config.alpha)
    print(f'T_n filas raw: {len(tn_df):,} | filas summary: {len(tn_summary)}')
else:
    print('T_n omitido.')

[T_n MC paralelo] 200 tareas en 11 workers (B=39, R=20)
[T_n MC paralelo] 100%  (200/200)  ETA: 0.0 min
[T_n MC paralelo] Completado en 0.4 min
T_n filas raw: 200 | filas summary: 10


## Ejecutar `S_n` con `argmin_pyomo`

In [5]:
sn_df = None
sn_summary = None

if run_sn:
    if use_parallel:
        from src.simulation_sn_parallel import run_simulation_sn_parallel
        n_workers = max(1, (os.cpu_count() or 2) - 1)
        sn_df = run_simulation_sn_parallel(specs, sn_config, n_workers=n_workers)
    else:
        from src.simulation_sn import run_simulation_sn
        sn_df = run_simulation_sn(specs, sn_config)

    sn_summary = summarize_sn(sn_df, alpha=sn_config.alpha)
    print(f'S_n filas raw: {len(sn_df):,} | filas summary: {len(sn_summary)}')
else:
    print('S_n omitido.')

[S_n MC paralelo] 100 tareas en 11 workers (B=39, R=10)
[S_n MC paralelo] 100%  (100/100)  ETA: 0.0 min
[S_n MC paralelo] Completado en 0.2 min
S_n filas raw: 100 | filas summary: 10


## Guardar resultados

In [6]:
data_dir = ROOT / 'results' / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

saved = []

if tn_df is not None:
    tn_raw_csv = data_dir / f'tn_pyomo_simulation_raw{suffix}.csv'
    tn_sum_csv = data_dir / f'tn_pyomo_simulation_summary{suffix}.csv'
    tn_df.to_csv(tn_raw_csv, index=False)
    tn_summary.to_csv(tn_sum_csv, index=False)
    saved.extend([tn_raw_csv, tn_sum_csv])

if sn_df is not None:
    sn_raw_csv = data_dir / f'sn_pyomo_simulation_raw{suffix}.csv'
    sn_sum_csv = data_dir / f'sn_pyomo_simulation_summary{suffix}.csv'
    sn_df.to_csv(sn_raw_csv, index=False)
    sn_summary.to_csv(sn_sum_csv, index=False)
    saved.extend([sn_raw_csv, sn_sum_csv])

print('Archivos guardados:')
for path in saved:
    print(f'  {path}')

Archivos guardados:
  c:\Users\juanc\github\Proyecto2EstadisticaNP\results\data\tn_pyomo_simulation_raw_quick.csv
  c:\Users\juanc\github\Proyecto2EstadisticaNP\results\data\tn_pyomo_simulation_summary_quick.csv
  c:\Users\juanc\github\Proyecto2EstadisticaNP\results\data\sn_pyomo_simulation_raw_quick.csv
  c:\Users\juanc\github\Proyecto2EstadisticaNP\results\data\sn_pyomo_simulation_summary_quick.csv


## Resumen rapido

In [7]:
if tn_summary is not None:
    print('T_n: tasas de rechazo')
    display(tn_summary.pivot_table(index=['dist', 'estimator'], columns='n', values='reject_rate').round(3))

if sn_summary is not None:
    print('S_n: tasas de rechazo')
    display(sn_summary.pivot_table(index=['dist', 'estimator', 'q', 'weight'], columns='n', values='reject_rate').round(3))

T_n: tasas de rechazo


,n,20,40
dist,estimator,,
"Cauchy(loc=2.0,scale=1.0)",argmin_pyomo,0.0,0.00
"Gamma(k=2.0,s=1.0)",argmin_pyomo,0.0,0.05
"Pareto(a=3.0,s=1.0)",argmin_pyomo,0.1,0.40
"Uniforme(1.0,3.0)",argmin_pyomo,0.0,0.10
"Weibull(k=1.5,s=1.0)",argmin_pyomo,0.0,0.10


S_n: tasas de rechazo


,,,n,20,40
dist,estimator,q,weight,,
"Cauchy(loc=2.0,scale=1.0)",argmin_pyomo,2,gauss_1.0,0.0,0.0
"Gamma(k=2.0,s=1.0)",argmin_pyomo,2,gauss_1.0,0.3,0.2
"Pareto(a=3.0,s=1.0)",argmin_pyomo,2,gauss_1.0,0.0,0.1
"Uniforme(1.0,3.0)",argmin_pyomo,2,gauss_1.0,0.0,0.1
"Weibull(k=1.5,s=1.0)",argmin_pyomo,2,gauss_1.0,0.0,0.2
